# Compare quantile regression across observation counts

Use the same synthetic data distribution, LP formulation, projection density rule,
and retrieval routines as the original m=5000 notebook. The main experiment uses
p=399, density=0.8, tau=0.2, and k=20% of m.

Retrieval retains the original line-based affine approximation and 30-step budget.
Feasibility is checked against the original constraints; stagnation is not success.
The original absolute L2 tolerance is retained, with RMS residual also reported.
Independent generation with the same seed does not create nested datasets.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# locate the project from either the notebook folder or project root
ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "quantile_regression.py").exists()
)
RESULTS_PATH = ROOT / "results" / "quantile_regression" / "comparison.csv"


# Run the experiment

Run this command in a terminal at the project root:

```text
.venv\Scripts\python.exe problems/quantile_regression/run_comparison.py
```

For repeated seeds, use a separate output file:

```text
.venv\Scripts\python.exe problems/quantile_regression/run_comparison.py --data-seeds 42 43 44 45 46 --projection-seeds 42 43 44 --output results/quantile_regression/repeated.csv
```

Each dataset/projection run includes an original solve and all three retrievals.
Existing completed runs are skipped when settings match. Use `--overwrite` to rerun.
Change RESULTS_PATH above to compare a different experiment.


In [ ]:
# load the saved comparison results
if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Run run_comparison.py first. Results were not found at {RESULTS_PATH}"
    )
results = pd.read_csv(RESULTS_PATH)
results["feasible"] = results["feasible"].astype(str).str.lower().eq("true")
results = results.sort_values(["m", "data_seed", "projection_seed", "method"])
print("Observation counts:", sorted(results["m"].unique()))
print("Number of comparison rows:", len(results))
display(results[["m", "p", "k", "data_seed", "projection_seed", "method", "stop_reason"]])


In [ ]:
# summarize runtime, objective quality, and original LP feasibility
summary = results.groupby(["m", "method"], as_index=False).agg(
    runs=("feasible", "size"),
    feasible_rate=("feasible", "mean"),
    median_pipeline_seconds=("time_pipeline", "median"),
    median_speedup=("speedup", "median"),
    median_feasible_objective_error=("relative_objective_error", "median"),
    median_equality_residual=("equality_residual", "median"),
    median_equality_residual_rms=("equality_residual_rms", "median"),
    max_bound_violation=("bound_violation", "max"),
    median_iterations=("iterations", "median"),
)
print("===== Comparison across observation counts =====")
display(summary.round(6))


In [ ]:
# compare complete pipelines with the direct original solve
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for method, group in summary.groupby("method"):
    group = group.sort_values("m")
    axes[0, 0].plot(group["m"], group["median_pipeline_seconds"], marker="o", label=method)
    if method != "Original LP":
        axes[0, 1].plot(group["m"], group["median_speedup"], marker="o", label=method)
        axes[1, 0].plot(group["m"], 100 * group["median_feasible_objective_error"], marker="o", label=method)
        axes[1, 1].plot(group["m"], 100 * group["feasible_rate"], marker="o", label=method)

axes[0, 0].set(title="Complete pipeline time", ylabel="Median seconds")
axes[0, 1].set(title="Direct / projected pipeline time", ylabel="Median speedup")
axes[0, 1].axhline(1.0, color="gray", linestyle="--")
axes[1, 0].set(title="Objective error at feasible retrieved points", ylabel="Median relative error (%)")
axes[1, 1].set(title="Original LP feasibility", ylabel="Successful runs (%)", ylim=(-5, 105))
for ax in axes.flat:
    ax.set_xlabel("Number of observations (m)")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
plt.show()
# Missing objective-error points mean there were no comparable feasible results.


In [ ]:
# inspect wall-clock components and retrieval stop reasons
components = results.groupby(["m", "method"], as_index=False)[[
    "time_original_solve", "time_T_generation", "time_projection",
    "time_TLP_solve", "time_retrieval", "time_pipeline",
]].median()
print("===== Wall-clock time (seconds) =====")
display(components.round(3))
print("===== Retrieval stop reasons =====")
display(results.groupby(["m", "method", "stop_reason"]).size().rename("runs").reset_index())


In [ ]:
# save the comparison summary and figure
summary_path = RESULTS_PATH.with_name(RESULTS_PATH.stem + "_summary.csv")
figure_path = RESULTS_PATH.with_name(RESULTS_PATH.stem + "_plots.png")
summary.to_csv(summary_path, index=False)
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
print("Summary saved to:", summary_path)
print("Figure saved to:", figure_path)
